# 🍼 Phase 1: Business Understanding
## STP Natality — Weight Analysis of Newborns

> **Purpose:** Document the business problem, dataset overview, and clinical context.  
> **Output:** Confirms project feasibility and kicks off Phase 2 (Data Engineering).  
> **Reference:** [`docs/business_understanding.md`](../docs/business_understanding.md)

---

## 0. Environment Setup

In [ ]:
import sys
import os
from pathlib import Path
import logging

# ── Ensure project root is on the Python path ──────────────────────────────
# This allows 'from src.data.ingest import ...' to work from inside notebooks/
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Plot style ─────────────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (10, 5),
    "font.size": 12,
})

print(f"pandas {pd.__version__} | numpy {np.__version__}")

---
## 1. Problem Statement

**Goal:** Predict the birth weight (in grams) of newborns from prenatal and demographic features available *before* delivery, to assist physicians in identifying pregnancies at risk of Low Birth Weight (LBW) or Macrosomia.

| Clinical Category | Threshold | Risk |
|-------------------|-----------|------|
| Very Low Birth Weight (VLBW) | < 1,500g | ⚠️ Severe |
| Low Birth Weight (LBW) | < 2,500g | ⚠️ High |
| **Normal** | **2,500g – 4,000g** | ✅ Optimal |
| Macrosomia | > 4,000g | ⚠️ High |

**ML Task:** Supervised Regression → predict `weight_grams` (continuous)  
**Success Metric:** MAE < 300g on held-out test set  
**Secondary Task:** LBW binary classification (sensitivity target: > 85%)

---
## 2. Dataset Quick-Look

We load a **small sample** (10,000 rows per file) to verify the schema and get initial statistics without reading all ~27M rows.

In [ ]:
from src.data.ingest import load_all_files, get_data_summary, NATALITY_FILES

# Load 10,000 rows from each of the 5 files (50,000 rows total — fast)
DATA_DIR = PROJECT_ROOT / "newborn_data"
SAMPLE_N = 10_000

print(f"Loading {SAMPLE_N:,} rows per file from: {DATA_DIR}")
print(f"Files to load: {list(NATALITY_FILES.keys())}")

In [ ]:
df_sample = load_all_files(DATA_DIR, nrows_per_file=SAMPLE_N)

summary = get_data_summary(df_sample)
print(f"\n📊 Sample Dataset Summary:")
print(f"  Rows:       {summary['n_rows']:,}")
print(f"  Columns:    {summary['n_cols']}")
print(f"  Files:      {summary['files_included']}")
print(f"  Year range: {summary['year_range']['min']} – {summary['year_range']['max']}")
print(f"  Memory:     {summary['memory_mb']:.1f} MB")

In [ ]:
# Schema overview
print("Column dtypes and null rates:")
null_pct = (df_sample.isnull().mean() * 100).round(1)
schema_df = pd.DataFrame({
    "dtype": df_sample.dtypes,
    "null_%": null_pct,
    "sample_value": df_sample.iloc[0],
})
schema_df

---
## 3. Target Variable: `weight_pounds` → `weight_grams`

In [ ]:
from src.data.preprocess import run_preprocessing

df_clean = run_preprocessing(df_sample)

print(f"After preprocessing: {len(df_clean):,} rows (from {len(df_sample):,})")
print(f"\nTarget variable ('weight_grams') statistics:")
df_clean["weight_grams"].describe().round(1)

In [ ]:
# ── Clinical threshold lines ───────────────────────────────────────────────
VLBW = 1500
LBW  = 2500
MACRO = 4000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Histogram ——————————————————————————————————————————————————————————————
ax = axes[0]
ax.hist(df_clean["weight_grams"].dropna(), bins=80, color="steelblue", alpha=0.85, edgecolor="white")
for thresh, label, color in [
    (VLBW,  "VLBW\n<1500g", "red"),
    (LBW,   "LBW\n<2500g",  "orange"),
    (MACRO, "Macrosomia\n>4000g", "purple"),
]:
    ax.axvline(thresh, color=color, linestyle="--", linewidth=1.8, label=label)
ax.set_title("Birth Weight Distribution (sample)", fontsize=13, fontweight="bold")
ax.set_xlabel("Weight (grams)")
ax.set_ylabel("Count")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# — Category pie ————————————————————————————————————————————————————————————
ax2 = axes[1]
w = df_clean["weight_grams"].dropna()
cats = pd.cut(
    w,
    bins=[-np.inf, VLBW, LBW, MACRO, np.inf],
    labels=["VLBW (<1500g)", "LBW (1500–2500g)", "Normal (2500–4000g)", "Macrosomia (>4000g)"],
)
counts = cats.value_counts().sort_index()
colors_pie = ["#d62728", "#ff7f0e", "#2ca02c", "#9467bd"]
ax2.pie(
    counts,
    labels=[f"{l}\n({v/len(w)*100:.1f}%)" for l, v in zip(counts.index, counts)],
    colors=colors_pie,
    startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
ax2.set_title("Birth Weight Category Distribution", fontsize=13, fontweight="bold")

plt.suptitle(f"Target Variable Analysis (n={len(w):,} records)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "diagrams" / "01_target_distribution.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("\nCategory counts:")
print(counts.to_string())

---
## 4. Feature Preview

In [ ]:
from src.data.feature_engineering import engineer_features

df_feat = engineer_features(df_clean)

# Show new columns added by feature engineering
original_cols = set(df_clean.columns)
new_cols = [c for c in df_feat.columns if c not in original_cols]
print(f"New features engineered ({len(new_cols)}): {new_cols}")

df_feat[new_cols + ["weight_grams"]].describe().round(2)

In [ ]:
# ── Top feature correlation with target ───────────────────────────────────
numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["weight_pounds", "source_year", "record_weight"]]

correlations = (
    df_feat[numeric_cols]
    .corr()["weight_grams"]
    .drop("weight_grams")
    .abs()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(correlations.index[::-1], correlations.values[::-1],
               color=plt.cm.Blues(np.linspace(0.4, 0.9, len(correlations))))
ax.set_xlabel("|Pearson Correlation| with weight_grams")
ax.set_title("Top 15 Features by Absolute Correlation with Birth Weight (sample)",
             fontsize=12, fontweight="bold")
for bar, val in zip(bars, correlations.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "diagrams" / "01_feature_correlations.png",
            bbox_inches="tight", dpi=150)
plt.show()

---
## 5. Temporal Coverage

In [ ]:
# Birth weight trend by year (using sample)
yearly = (
    df_feat.groupby("year_fix")["weight_grams"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly["year_fix"], yearly["mean"], marker="o", linewidth=2,
        color="steelblue", label="Mean birth weight")
ax.fill_between(
    yearly["year_fix"],
    yearly["mean"] - yearly["std"],
    yearly["mean"] + yearly["std"],
    alpha=0.2, color="steelblue", label="± 1 SD"
)
# Split lines
ax.axvline(2010, color="orange", linestyle="--", linewidth=1.5, label="Val split (2010)")
ax.axvline(2016, color="red",    linestyle="--", linewidth=1.5, label="Test split (2016)")
ax.axhline(LBW,  color="gray",   linestyle=":",  linewidth=1.2, label="LBW threshold (2500g)")
ax.set_xlabel("Year")
ax.set_ylabel("Birth Weight (grams)")
ax.set_title("Mean Birth Weight by Year with Train/Val/Test Split Boundaries",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}g"))
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "diagrams" / "01_temporal_trend.png",
            bbox_inches="tight", dpi=150)
plt.show()

---
## 6. Temporal Split Preview

In [ ]:
from src.data.split import temporal_split

train, val, test = temporal_split(df_feat)

total = len(df_feat)
print("\n📊 Split Summary (on sample data):")
print(f"{'Split':<8} {'Rows':>8} {'%':>6} {'Year Range'}")
print("-" * 40)
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    yr_min = int(split["year_fix"].min()) if len(split) > 0 else "—"
    yr_max = int(split["year_fix"].max()) if len(split) > 0 else "—"
    print(f"{name:<8} {len(split):>8,} {100*len(split)/total:>5.1f}%  {yr_min}–{yr_max}")

---
## 7. Missing Value Overview

In [ ]:
import missingno as msno

# Show missing value pattern for the clean sample
fig, ax = plt.subplots(figsize=(14, 6))
msno.bar(df_clean, ax=ax, color="steelblue", fontsize=9)
ax.set_title("Data Completeness by Column (after preprocessing)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "docs" / "diagrams" / "01_missing_values.png",
            bbox_inches="tight", dpi=150)
plt.show()

# Table view
missing_df = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_%": (df_clean.isnull().mean() * 100).round(2),
}).query("missing_count > 0").sort_values("missing_%", ascending=False)
print("\nColumns with missing values:")
print(missing_df.to_string())

---
## 8. Business Understanding Summary

| Item | Decision |
|------|----------|
| **ML Task** | Supervised Regression (`weight_grams`) |
| **Primary metric** | MAE < 300g |
| **LBW sensitivity target** | > 85% |
| **Features excluded (leakage)** | `apgar_1min`, `apgar_5min`, `day` |
| **Split strategy** | Temporal: train <2010, val 2010–2015, test ≥2016 |
| **Development sample size** | 10K rows/file (50K total) for fast iteration |
| **Production training** | Full 27M rows with Polars/chunked Pandas |
| **Data ethics** | De-identified public dataset; no GDPR issues for research |
| **System role** | Decision **support** only — clinician always has final authority |

✅ **Phase 1 complete. Ready to proceed to Phase 2: Data Engineering.**